# 📗 사람이 검토하는 안내문 그래프를 만듭니다

**서비스 점검 안내문 작성 에이전트**를 만듭니다. 점검 정보를 바탕으로 GPT가 초안을 작성하고, 사람의 수정 의견을 반영해 **승인된 안내문을 확정**합니다. 작성 과정과 진행 상태는 스트리밍으로 보여 줍니다.

<img src="images/02_hitl_overview.png" width="1050" style="background:white" alt="서비스 점검 안내문 작성 에이전트: GPT가 초안을 작성하고 사람이 검토합니다. 승인하면 확정, 수정 요청이면 다시 작성·검토, 반려하면 종료합니다.">

## 실습의 목표

1. 검토에 필요한 State를 정의합니다.
2. 초안 작성·사람 검토·수정·확정 노드를 만듭니다.
3. 검토 결과에 따른 조건부 엣지와 수정 사이클을 연결합니다.
4. `interrupt`와 `Command(resume=...)`로 대기·재개합니다.
5. 진행 상태와 생성되는 글을 스트리밍으로 확인합니다.

체크포인터에 저장된 **검토 대기 상태를 사람의 응답으로 재개**합니다.

## 실행을 준비합니다

이 노트북은 위에서부터 독립적으로 실행합니다. 앞 단원과 같은 `.env`의 `OPENAI_API_KEY`와 `OPENAI_CHAT_MODEL=gpt-5.6-luna`를 사용합니다. 초안 작성과 LLM 수정에 각각 GPT를 한 번 호출합니다. 승인·반려·직접 편집은 추가 모델 호출이 없습니다.

In [ ]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

# 앞 단원과 같은 환경 설정을 읽고 답변 모델을 준비
material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)

## 1. State에 검토할 문서와 사람의 선택을 담습니다

**HITL**(Human-in-the-loop)은 자동 실행 도중 사람의 판단을 받아 다음 작업을 정하는 방식입니다. State에는 노드 사이에서 전달할 원문·초안·검토 결과를 둡니다. 노드가 반환한 필드는 새 값으로 교체되고, 반환하지 않은 필드는 유지됩니다.

State 딕셔너리의 **키는 필드 이름**, **값은 현재 데이터**입니다. 예를 들어 `draft`의 값은 검토 중인 안내문 문자열입니다.

In [ ]:
from typing import Literal, TypedDict

# 노드들이 공유할 검토 상태 정의
class NoticeState(TypedDict):
    brief: str  # 초안과 수정에 사용할 확정된 점검 정보
    draft: str  # 사람이 읽고 검토할 최신 안내문
    decision: str  # approve(승인), edit(수정 요청), reject(반려) 중 선택한 값
    feedback: str  # 수정 노드에 전달할 사람의 요청
    published_text: str  # 승인 후 결과로 확인할 확정 안내문

# 학습용 가상 서비스의 점검 정보로 초기 상태 선언
notice_input: NoticeState = {
    "brief": """서비스: 모아 예약
점검: 2026년 10월 7일 02:00~03:00, 한국 시간
영향: 점검 중 신규 예약과 예약 변경을 이용할 수 없음
기존 예약 내역 조회는 가능
문의: 고객센터의 1:1 문의
보상 여부나 점검 연장에 관한 확정 내용은 없음""",
    "draft": "",
    "decision": "",
    "feedback": "",
    "published_text": "",
}
print(notice_input["brief"])

## 2. 작업별 노드를 만듭니다

### 1번 write_draft: 안내문 초안을 작성합니다

`brief`를 읽어 `draft`를 채웁니다. `get_stream_writer()`는 진행 정보를 보낼 함수인 `writer`를 반환합니다. `writer(값)`으로 보낸 정보는 5절의 `custom` 스트림에서 받습니다. 아래에서는 함수만 정의하고, 연결한 그래프에서 실행합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langgraph.config import get_stream_writer

# 확정 정보만 사용하도록 안내문 생성 기준 제공
draft_prompt = ChatPromptTemplate.from_messages([
    ("system", "서비스 점검 안내문을 한국어로 작성하세요. "
     "제목, 점검 시간, 이용 제한, 이용 가능 기능, 문의 경로를 짧게 구분하세요. "
     "제공 정보에 없는 보상이나 완료 약속을 만들지 마세요. "
     "이용자가 미리 준비할 사항을 단정적으로 추가하지 마세요."),
    ("human", "확정된 정보:\n{brief}"),
])
draft_chain = draft_prompt | llm


def write_draft(state: NoticeState):
    """확정된 점검 정보로 검토할 초안을 만듭니다."""
    writer = get_stream_writer()  # custom 스트림으로 진행 정보를 보낼 함수
    writer({"node": "write_draft", "status": "안내문 작성 중"})
    response = draft_chain.invoke({"brief": state["brief"]})
    return {"draft": response.text}

### 2번 human_review: 초안을 보여 주고 사람의 결정을 기다립니다

`interrupt(검토 요청)`은 초안과 선택지를 호출자에게 전달하고 실행을 멈춥니다. 같은 thread_id로 `Command(resume=검토 응답)`을 보내면, 재실행된 노드의 interrupt가 그 응답을 반환합니다.

| 사람이 보낼 응답의 키 | 담을 값 |
|---|---|
| `action` | `approve`(승인), `edit`(LLM 수정 요청), `reject`(반려) |
| `feedback` | LLM에 전달할 수정 의견. 수정 요청일 때 사용 |
| `edited_text` | 사람이 직접 고친 최종 문장. 직접 편집 후 승인할 때 사용 |

`actions`는 보여 줄 선택지이며 응답을 자동 검사하지 않습니다. 실제 화면에서는 선택값과 입력 형식을 확인한 뒤 재개합니다.

대기하려면 **체크포인터와 `thread_id`가 필요**합니다. 잠시 멈춘 실행을 저장하고, 어느 실행을 이어 갈지 찾아야 하기 때문입니다.

<img src="images/04_interrupt_resume.png" width="1050" style="background:white" alt="interrupt로 초안과 선택지를 전달하고 대기한 뒤 같은 thread_id로 사람의 결정을 받아 노드를 처음부터 재개합니다.">

In [ ]:
from langgraph.types import Command, interrupt


def human_review(state: NoticeState):
    """초안을 전달해 대기하고, 사람의 선택과 수정 내용을 반환합니다."""
    print("검토 노드 시작")  # 재개할 때도 이 줄부터 다시 실행되는지 확인
    decision = interrupt({
        "stage": "notice_review",  # 화면에서 안내문 검토 단계임을 구분할 이름
        "draft": state["draft"],
        "actions": ["approve", "edit", "reject"],
        "guide": "approve: 승인, edit: 수정 요청, reject: 반려",
    })  # 초안과 선택지를 전달하고 중단. 재개 시 사람의 응답 딕셔너리를 반환
    return {
        "decision": decision["action"],  # 응답 전체에서 선택한 경로만 State에 저장
        "feedback": decision.get("feedback", ""),
        # 사람이 최종 문장을 직접 고쳤다면 그 내용을 새 초안으로 반영
        "draft": decision.get("edited_text", state["draft"]),
    }

재개하면 **`human_review`의 처음부터 다시 실행**합니다. 같은 interrupt에 도달하면 이번에는 전달받은 응답을 반환하고 아래 코드를 계속 실행합니다. 완료된 write_draft는 다시 실행하지 않습니다.

외부 발송·결제처럼 중복 실행되면 안 되는 일은 검토 노드 앞부분에 두지 않습니다. 승인 이후의 별도 노드에 배치하고, 실제 서비스에서는 재시도에도 중복 처리되지 않도록 설계합니다. `interrupt()`를 `try/except`로 감싸거나 실행마다 호출 순서를 바꾸지 않습니다.

### 3번 revise_draft: 검토 의견을 반영합니다

`brief`, `draft`, `feedback`을 읽어 새 `draft`를 만듭니다. decision을 빈 문자열로 바꾸고 수정한 초안을 다시 검토받습니다. 모델이 고쳤다는 이유만으로 자동 승인하지 않습니다.

In [ ]:
# 사람의 의견을 반영하되 점검 정보는 바꾸지 않도록 수정 기준 제공
revise_prompt = ChatPromptTemplate.from_messages([
    ("system", "서비스 점검 안내문을 검토 의견에 따라 수정하세요. "
     "원문의 확정된 시간, 이용 제한, 이용 가능 기능, 문의 경로를 보존하세요. "
     "제공되지 않은 정책이나 약속을 추가하지 말고 수정된 안내문만 출력하세요."),
    ("human", "확정 정보:\n{brief}\n\n현재 초안:\n{draft}\n\n검토 의견:\n{feedback}"),
])
revise_chain = revise_prompt | llm


def revise_draft(state: NoticeState):
    """확정 정보와 검토 의견으로 초안을 고쳐 다시 검토받습니다."""
    response = revise_chain.invoke({
        "brief": state["brief"],
        "draft": state["draft"],
        "feedback": state["feedback"],
    })
    return {"draft": response.text, "decision": ""}

### 4번 publish: 승인된 안내문을 확정합니다

`draft`를 `published_text`로 저장합니다. 이 실습의 확정은 State 안에서 결과를 보관하는 일입니다.

In [ ]:
def publish(state: NoticeState):
    """승인받은 초안을 확정 결과로 저장합니다."""
    # 승인된 초안을 published_text에 복사해 확정 결과로 반환
    return {"published_text": state["draft"]}

## 3. 조건부 엣지와 수정 사이클을 연결합니다

### 조건부 엣지 A: 사람의 선택으로 경로를 정합니다

| 2번 검토 결과 | 다음 노드 | 이어지는 흐름 |
|---|---|---|
| `approve` | 4번 `publish` | 확정 후 종료 |
| `edit` | 3번 `revise_draft` | 수정 후 2번 재검토 |
| `reject` | `END` | 확정하지 않고 종료 |

라우팅 함수는 저장된 선택을 읽습니다. 이 선택을 판단하기 위해 모델을 다시 호출하지 않습니다. 예제는 `approve`와 `edit` 이외의 값을 반려로 처리합니다.

In [ ]:
def route_review(state: NoticeState) -> Literal["approve", "edit", "reject"]:
    """사람의 선택에 따라 확정, 수정, 종료 경로를 반환합니다."""
    # 사람의 선택을 경로 라벨로 바꾸며 모델은 다시 호출하지 않음
    if state["decision"] == "approve":
        return "approve"
    if state["decision"] == "edit":
        return "edit"
    return "reject"

### 노드·엣지를 등록하고 실행 그래프를 만듭니다

조건부 엣지의 경로 딕셔너리는 **키가 라우팅 함수의 반환값, 값이 다음 노드 이름**입니다. 목적지를 명시하면 그래프 이미지에서도 분기와 수정 사이클을 확인할 수 있습니다.

| 구분 | 역할 |
|---|---|
| 체크포인트 | 실행 단계별 State와 다음 작업 등을 담은 저장 기록 |
| 체크포인터 | 체크포인트를 저장하고 읽는 객체. 여기서는 `InMemorySaver` 사용 |
| `thread_id` | 어느 대화·작업의 기록을 사용할지 정하는 ID |

`compile(checkpointer=...)`로 저장 기능을 연결하고, 실행할 때 `config`의 `thread_id`로 작업을 구분합니다. **체크포인터 없이 `thread_id`만 지정하면 상태가 자동 저장되지 않습니다.**

`InMemorySaver`의 기록은 해당 객체의 메모리에 남습니다. 커널 재시작 뒤에도 복원하는 파일 저장 방식은 다음 교안에서 다룹니다.

In [ ]:
from uuid import uuid4
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from IPython.display import Image, Markdown, display

# 노드 등록과 실행 흐름 연결
builder = StateGraph(NoticeState)
builder.add_node("write_draft", write_draft)
builder.add_node("human_review", human_review)
builder.add_node("revise_draft", revise_draft)
builder.add_node("publish", publish)
builder.add_edge(START, "write_draft")
builder.add_edge("write_draft", "human_review")
builder.add_conditional_edges("human_review", route_review, {
    "approve": "publish",
    "edit": "revise_draft",
    "reject": END,
})
builder.add_edge("revise_draft", "human_review")
builder.add_edge("publish", END)

# 대기 중인 실행 상태를 보관할 체크포인터 연결
checkpointer = InMemorySaver()
notice_graph = builder.compile(checkpointer=checkpointer)
display(Image(notice_graph.get_graph().draw_mermaid_png()))

## 4. 멈춘 실행에 사람의 결정을 전달합니다

### 초안을 생성하고 검토 지점에서 멈춥니다

같은 체크포인터에서 같은 `thread_id`를 쓰면 이전 State와 대기 정보를 읽습니다. `uuid4()`로 새 문서의 ID를 만들고, 수정·승인 시에는 생성한 설정을 그대로 재사용합니다. 이 예제의 `thread_id`는 문서별 검토 기록을 구분합니다.

`invoke()` 결과의 `__interrupt__`에 대기 중인 요청이 담깁니다. 저장 후에는 `get_state(config).interrupts`로 읽습니다. 목록에서 꺼낸 중단 객체를 `item`이라고 하면 다음처럼 구분합니다.

| 항목 | 역할 |
|---|---|
| `thread_id` | 전체 문서·작업을 식별합니다. 재개할 때 유지합니다. |
| `item.id` | LangGraph가 부여한 중단 ID입니다. 응답할 요청을 지정합니다. |
| `item.value["stage"]` | 우리가 정한 검토 단계 이름입니다. 화면에 어떤 검토인지 표시합니다. |

**여러 검토가 있으면 현재 요청이 내용 검토인지 최종 승인인지 알아야 합니다.** `stage`는 표시·업무 구분용 데이터이며, 응답은 `Command(resume={중단_ID: 응답})`으로 연결합니다. 한 요청만 대기할 때는 기존의 `Command(resume=응답)`도 가능합니다. 아래에서 두 방식을 확인합니다.

초안 작성과 첫 수정에 GPT를 총 2회 호출하며, 추가 수정 요청마다 1회 더 호출합니다. 아래 반려와 직접 편집 예시는 새 초안을 만들 때 각각 1회 호출합니다.

In [ ]:
# 검토할 문서의 실행 기록을 식별할 설정
notice_config = {"configurable": {"thread_id": f"notice-{uuid4()}"}}
paused = notice_graph.invoke(notice_input, notice_config)

# 중단되면 __interrupt__에 요청 목록이 담김. 없으면 빈 튜플로 읽기
interrupts = paused.get("__interrupt__", ())
print("사람의 응답 대기:", bool(interrupts))
if interrupts:
    review_interrupt = interrupts[0]  # 이 그래프는 한 번에 한 요청만 기다림
    review_request = review_interrupt.value
    print("중단 ID:", review_interrupt.id)
    print("검토 단계:", review_request["stage"])
    print("검토 요청:", review_request["guide"])
    display(Markdown(review_request["draft"]))
print("아직 확정되지 않은 결과:", paused["published_text"])

### 수정 요청으로 재개하고 새 초안을 다시 검토합니다

수정 의견을 현재 중단 ID에 연결해 전달합니다. 수정 후에는 같은 `notice_review` 단계에서 새 검토를 기다립니다. `stage`는 같아도 이전 중단 ID를 재사용하지 않고, 현재 대기 목록에서 ID를 다시 읽습니다. 4절 마지막에서 수정본을 직접 검토합니다.

In [ ]:
# 키는 검토 항목 이름, 값은 사람이 선택하거나 작성한 내용
review_response = {
    "action": "edit",
    "feedback": "기존 예약 내역 조회는 가능하다는 내용을 첫 문장에 강조해 주세요.",
}
# 현재 대기 요청의 ID를 읽어 그 요청에 수정 의견 전달
review_interrupt = notice_graph.get_state(notice_config).interrupts[0]
revised = notice_graph.invoke(
    Command(resume={review_interrupt.id: review_response}), notice_config,
)
# 수정된 안내문을 확인한 뒤, 4절 마지막의 직접 검토에서 재개
revised_requests = revised.get("__interrupt__", ())
for item in revised_requests:
    print("중단 ID:", item.id)
    print("검토 단계:", item.value["stage"])
    display(Markdown(item.value["draft"]))
print("수정 완료. 4절 마지막에서 승인·수정·반려를 직접 선택하세요.")
print("다음 실행 지점:", notice_graph.get_state(notice_config).next)

### 반려하면 확정 결과를 만들지 않습니다

새 문서 ID로 독립적인 검토를 시작합니다. `published_text`가 빈 문자열이고 남은 실행 지점이 없으면, 초안만 남긴 채 종료된 것입니다.

In [ ]:
# 별도 문서에서 반려 경로 확인: 새 초안 생성에 GPT 1회 호출
reject_config = {"configurable": {"thread_id": f"reject-{uuid4()}"}}
rejected_pause = notice_graph.invoke(notice_input, reject_config)
rejected = notice_graph.invoke(Command(resume={"action": "reject"}), reject_config)
print("검토 결과:", rejected["decision"])
print("확정 안내문:", repr(rejected["published_text"]))
print("남은 실행 지점:", notice_graph.get_state(reject_config).next)

### 사람이 고친 문장을 그대로 승인할 수도 있습니다

`edited_text`에 직접 편집한 문장을 전달하면 모델을 다시 호출하지 않고 그 문장을 확정합니다. `edit`은 **LLM 수정 요청**, `approve`와 `edited_text`의 조합은 **사람이 편집한 문장 승인**입니다.

In [ ]:
# 새 초안 생성 뒤 사람이 문장을 직접 편집해 승인
manual_config = {"configurable": {"thread_id": f"manual-{uuid4()}"}}
manual_pause = notice_graph.invoke(notice_input, manual_config)
reviewed_text = """모아 예약 점검 안내
기존 예약 내역 조회는 점검 중에도 가능합니다.
2026년 10월 7일 02:00~03:00(한국 시간)에는 신규 예약과 예약 변경을 이용할 수 없습니다.
문의는 고객센터의 1:1 문의를 이용해 주세요."""
manual_result = notice_graph.invoke(
    Command(resume={"action": "approve", "edited_text": reviewed_text}),
    manual_config,
)
display(Markdown(manual_result["published_text"]))

**`Command`의 두 용도:** `Command(resume=값)`은 그래프 밖에서 대기 중인 실행에 답을 전달합니다. 노드가 반환하는 `Command(update=갱신값, goto="다음 노드")`는 상태 갱신과 다음 이동을 함께 지정할 때 사용합니다. 이 그래프는 흐름을 이미지로 읽기 쉽도록 `add_conditional_edges`로 이동을 정의했습니다.

[interrupt와 재개 공식 문서](https://docs.langchain.com/oss/python/langgraph/interrupts)

### 직접 검토: 앞에서 수정한 안내문을 확인하고 선택합니다

앞에서 수정 후 대기시킨 `notice_config`의 안내문을 다시 읽습니다. 별도 문서로 진행한 반려·직접 편집 예시와 구분하세요.

안내문을 확인한 뒤 `승인`, `수정`, `반려` 중 하나를 입력하세요. `수정`을 선택하면 의견을 받아 다시 작성하고, 새 안내문을 보여 준 뒤 다시 선택을 기다립니다. **승인을 선택했을 때만 확정문이 저장됩니다.**

In [ ]:
# 현재 저장된 검토 요청을 읽어, 안내문을 보여 준 뒤 사람의 선택을 받기
snapshot = notice_graph.get_state(notice_config)
action_map = {"승인": "approve", "수정": "edit", "반려": "reject"}

while snapshot.interrupts:
    review_interrupt = snapshot.interrupts[0]
    review_request = review_interrupt.value
    print("중단 ID:", review_interrupt.id)
    print("검토 단계:", review_request["stage"])
    display(Markdown("### 검토할 안내문"))
    display(Markdown(review_request["draft"]))
    choice = input("안내문을 확인한 뒤 선택하세요 [승인 / 수정 / 반려]: ").strip()
    if choice not in action_map:
        print("승인, 수정, 반려 중 하나를 입력하세요.")
        continue

    # 화면에서 선택한 값으로 재개하며, 수정 요청일 때만 의견을 함께 전달
    review_response = {"action": action_map[choice]}
    if choice == "수정":
        feedback = input("어떤 내용을 수정할까요? ").strip()
        if not feedback:
            print("수정할 내용을 입력해야 다시 작성할 수 있습니다.")
            continue
        review_response["feedback"] = feedback

    notice_graph.invoke(
        Command(resume={review_interrupt.id: review_response}), notice_config,
    )
    snapshot = notice_graph.get_state(notice_config)  # 추가 수정이면 새 검토 요청이 생김

# 승인·반려 후에는 대기가 끝나며, 승인된 문서만 확정 결과로 표시
review_result = snapshot.values
print("검토 결과:", review_result["decision"])
if review_result["decision"] == "approve":
    display(Markdown("### 확정 안내문"))
    display(Markdown(review_result["published_text"]))
else:
    print("반려하여 안내문을 확정하지 않았습니다.")
print("남은 실행 지점:", snapshot.next)

## 5. 실행 중인 결과를 스트리밍으로 받습니다

**스트리밍**은 실행 중 생기는 결과를 차례로 받는 방식입니다. 사용자는 전체 작업이 끝나기 전에도 생성되는 답변과 현재 진행 상황을 확인할 수 있습니다.

<img src="images/11_streaming_why_where.png" width="1050" style="background:white" alt="완료 후 한 번에 받는 방식과 실행 중 결과를 받는 스트리밍을 비교합니다. 챗봇·문서 작성, 검색·에이전트 작업, 사람의 검토 대기 화면에 활용합니다.">

이번 실습에서는 **작성 중인 문장**과 **사람의 검토를 기다리는 상태**를 화면에 표시합니다. `stream()`도 그래프를 실행하므로 새 입력으로 호출하면 LLM도 다시 호출됩니다.

### 무엇을 볼지 모드로 고릅니다

| 모드 | 받는 데이터 | 사용할 때 |
|---|---|---|
| `updates` | `{노드 이름: 변경한 필드}` | 어느 노드가 무엇을 바꿨는지 확인 |
| `values` | 초기 상태와 단계별 전체 State | 현재 초안·검토 결과를 함께 확인 |
| `messages` | `(메시지 조각, 실행 정보)` | 생성 중인 문장을 화면에 표시 |
| `custom` | `writer(값)`으로 보낸 데이터 | “안내문 작성 중” 같은 진행 알림 |

`messages`는 모델 출력을 받으므로 State에 `messages` 필드가 없어도 됩니다. `custom` 알림은 State를 바꾸지 않습니다.

<details><summary>추가 진단 모드 3가지</summary>

| 모드 | 확인할 내용 |
|---|---|
| `checkpoints` | 저장된 상태·설정·다음 작업. 체크포인터 필요 |
| `tasks` | 작업 시작·종료, 결과와 오류 |
| `debug` | 작업·체크포인트의 상세 정보. 출력량이 많음 |

저장 관련 이벤트는 체크포인터를 연결했을 때 받습니다.

</details>

<img src="images/05_streaming_views.png" width="1050" style="background:white" alt="같은 그래프의 실행을 노드 갱신, 전체 상태, 메시지 조각, 직접 보낸 진행 정보로 관찰합니다.">

### v2와 v3는 결과를 읽는 방법이 다릅니다

| 사용 방법 | 결과의 모양 | 읽는 방법 |
|---|---|---|
| `stream(..., version="v2")` | `type`, `data`, `ns`가 있는 딕셔너리 | `type`으로 종류를 구분하고 `data`를 읽기 |
| `stream_events(..., version="v3")` | 실행 객체 `run` | `run.messages`, `run.values`, `run.output`에서 필요한 결과 읽기 |

**v2는 결과의 종류를 직접 구분하고, v3는 필요한 결과를 속성으로 골라 읽습니다.** 두 메서드의 출력 규약을 가리키며, v3 API는 현재 실험 단계입니다.

### stream: type으로 구분하고 data를 읽습니다

`stream(..., version="v2")`는 모드 수와 관계없이 같은 형식의 **StreamPart 딕셔너리**를 내보냅니다.

- `type`: 모드 이름
- `data`: 위 표에 정리한 해당 모드의 데이터
- `ns`: 하위 그래프 경로. 이번 그래프에서는 빈 튜플

아래에서는 네 모드를 **한 번의 실행**에서 받습니다. 답변·진행·State를 별도 영역에 표시해 문장 사이에 로그가 섞이지 않게 합니다. `display_id=True`로 만든 영역은 `.update()`로 같은 자리에 갱신합니다. 노드가 `llm.invoke()`를 호출해도 스트리밍을 지원하는 모델이면 `messages`로 생성 조각을 받을 수 있습니다.

In [ ]:
# 새 문서의 검토를 시작하고 답변·진행·State 표시 영역을 분리
stream_config = {"configurable": {"thread_id": f"stream-{uuid4()}"}}
progress_display = display(Markdown("**진행:** 작성 준비"), display_id=True)
state_display = display({"초안 글자 수": 0, "검토 선택": "선택 전"}, display_id=True)
answer_display = display(Markdown(""), display_id=True)
message_parts = []

for part in notice_graph.stream(
    notice_input,
    stream_config,
    stream_mode=["updates", "values", "messages", "custom"],
    version="v2",
):
    if part["type"] == "messages":
        message, metadata = part["data"]  # 메시지 조각과 호출 노드 등의 실행 정보
        if metadata.get("langgraph_node") == "write_draft" and message.text:
            message_parts.append(message.text)  # 빈 조각은 제외하고 받은 문장을 누적
            answer_display.update(Markdown("".join(message_parts)))
    elif part["type"] == "updates":
        update = part["data"]
        if "__interrupt__" in update:
            progress_display.update(Markdown("**진행:** 사람의 검토 대기"))
        else:
            progress_display.update(Markdown("**완료한 노드:** " + ", ".join(update)))
    elif part["type"] == "values":
        state = part["data"]  # 전체 State 중 화면에서 확인할 항목만 표시
        state_display.update({
            "초안 글자 수": len(state["draft"]),
            "검토 선택": state["decision"] or "선택 전",
        })
    elif part["type"] == "custom":
        progress_display.update(Markdown("**진행:** " + part["data"]["status"]))

# 같은 실행의 저장 결과로 완성된 문서를 확인. 모델을 다시 호출하지 않음
stream_snapshot = notice_graph.get_state(stream_config)
answer_display.update(Markdown(stream_snapshot.values["draft"]))
print("사람의 응답 대기:", bool(stream_snapshot.interrupts))
for item in stream_snapshot.interrupts:
    print("중단 ID:", item.id)
    print("검토 단계:", item.value["stage"])

`updates`의 `__interrupt__`는 사람의 검토를 기다린다는 알림입니다. 같은 `thread_id`로 재개합니다. 아래 스트리밍 시연에서는 검토자가 승인한 응답을 코드로 전달합니다. 직접 입력하는 흐름은 4절 마지막에서 작성했습니다.

In [ ]:
# 스트리밍으로 시작한 검토도 같은 ID로 재개
stream_result = notice_graph.invoke(
    Command(resume={"action": "approve"}), stream_config,
)
print("검토 결과:", stream_result["decision"])
display(Markdown(stream_result["published_text"]))

### stream_events: 필요한 결과를 run에서 읽습니다

`stream_events(..., version="v3")`는 `run`이라는 **실행 객체**를 반환합니다. `stream`처럼 `type`으로 나누는 대신, 결과별 속성인 **typed projection**으로 읽습니다. `stream_mode`는 지정하지 않습니다.

| 접근 | 읽는 내용 |
|---|---|
| `run.messages` | 메시지 스트림. 각 메시지의 `.text`를 순회해 문장 조각 읽기 |
| `run.values` | 단계별 전체 State |
| `run.output` | 실행 완료 또는 interrupt 대기 시점의 결과 |
| `run.interrupted` / `run.interrupts` | 사람의 응답 대기 여부 / 중단 요청 목록 |

`run.interleave("values", "messages")`로 상태와 문장을 처음부터 함께 읽습니다. 동기 실행은 순회하거나 `run.output`을 읽을 때 진행됩니다.

공식 가이드는 새 앱에 이 API를 권장하지만, **v3는 실험 단계**입니다. 이 교안의 고정 버전을 사용하세요. 두 예제는 서로 다른 `thread_id`로 각각 실행합니다.

In [ ]:
# 별도 검토 실행의 진행 상태·State·문장 영역 준비
projection_config = {"configurable": {"thread_id": f"projection-{uuid4()}"}}
progress_display = display(Markdown("**진행:** 작성 중"), display_id=True)
state_display = display({"초안 글자 수": 0, "검토 선택": "선택 전"}, display_id=True)
answer_display = display(Markdown(""), display_id=True)
run = notice_graph.stream_events(notice_input, projection_config, version="v3")

# 실행 시작부터 상태와 메시지를 함께 읽어 각각의 표시 영역 갱신
for kind, item in run.interleave("values", "messages"):
    if kind == "values":
        state_display.update({
            "초안 글자 수": len(item["draft"]),
            "검토 선택": item["decision"] or "선택 전",
        })
    elif kind == "messages":
        text_parts = []  # 새 모델 응답마다 누적할 문장 조각 초기화
        for text in item.text:
            text_parts.append(text)
            answer_display.update(Markdown("".join(text_parts)))

# 같은 실행의 최종 출력과 대기 여부 확인
projection_output = run.output
answer_display.update(Markdown(projection_output["draft"]))
progress_display.update(Markdown("**진행:** " + ("사람의 검토 대기" if run.interrupted else "완료")))
for item in run.interrupts:
    print("중단 ID:", item.id)
    print("검토 단계:", item.value["stage"])
    print("검토 요청:", item.value["guide"])

In [ ]:
# event streaming으로 멈춘 실행도 같은 ID와 Command로 재개
resumed_run = notice_graph.stream_events(
    Command(resume={"action": "approve"}), projection_config, version="v3",
)
projection_result = resumed_run.output  # 결과를 읽으면 승인 뒤 남은 실행이 진행됨
display(Markdown(projection_result["published_text"]))

[stream 모드](https://docs.langchain.com/oss/python/langgraph/streaming) · [event streaming](https://docs.langchain.com/oss/python/langgraph/event-streaming) · [출력 영역 갱신](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#IPython.display.DisplayHandle)

### ✅ 바로 확인 퀴즈

1. 수정한 문서를 사람이 다시 확인하게 만드는 연결은 어디인가요?
2. `Command(resume=...)`로 재개할 때 초안 작성 노드도 다시 실행될까요?
3. `values`와 `messages`는 어떤 화면에 각각 유용할까요?
4. `thread_id`, 중단 ID, `stage`는 각각 무엇을 구분하나요?

<details><summary>정답 보기</summary>

1. `revise_draft`에서 `human_review`로 돌아가는 엣지입니다.
2. 대기 중인 `human_review`가 처음부터 다시 실행됩니다. 완료된 `write_draft`는 다시 실행되지 않습니다.
3. `values`는 상태 확인 화면, `messages`는 생성 중인 문장 표시 화면에 유용합니다.
4. 전체 작업, 응답할 중단 요청, 화면에 표시할 검토 단계를 구분합니다.

</details>

## 🖐️ 함께 따라하기: 두 번의 검토를 거쳐 채용공고를 확정합니다

채용 담당자는 **내용이 정확한지**, 게시 책임자는 **이 공고를 게시해도 되는지** 판단합니다. 내용 승인이 게시 허가까지 뜻하지 않으므로 HITL을 두 노드로 나눕니다. 두 단계를 모두 승인해야 `approved_post`에 확정문이 남습니다. 실제 채용 사이트에는 게시하지 않습니다.

**공고 작성 → 내용 검토(HITL 1) → 게시 승인(HITL 2) → 공고 확정** 순서입니다. 내용 수정은 첫 검토로 돌아가며, 어느 단계든 반려하면 종료합니다.

프롬프트와 긴 입력 문자열은 제공합니다. State·노드·엣지를 작성한 뒤, 현재 중단 ID와 `stage`를 확인하며 차례로 재개하세요.

<img src="images/12_practice_two_hitl.png" width="1050" style="background:white" alt="내용 검토에서 수정과 재검토를 거친 뒤 게시 승인에서 다시 대기합니다. 두 승인이 모두 있어야 공고를 확정하며 어느 단계든 반려하면 종료합니다.">

In [ ]:
# 학습용 가상 채용공고의 확정 요건
job_requirements = """직무: 주니어 데이터 분석가
업무: 고객 문의 데이터를 정리하고 주간 지표 보고서를 작성
필수 역량: Python 기초, SQL 기초
근무지: 서울, 주 3일 사무실 근무와 주 2일 원격 근무
고용 형태: 정규직
지원 방법: 채용 페이지에서 이력서와 분석 프로젝트 1개 제출
급여 범위와 채용 마감일은 아직 확정하지 않음"""
print(job_requirements)

### 1. State와 초기 입력을 만듭니다

`JobState`를 `TypedDict`로 정의합니다. 모든 필드는 `str`입니다.

| 필드 | 담을 데이터 |
|---|---|
| `requirements` | 공고 작성과 수정에 사용할 확정 요건 |
| `post` | 검토할 최신 공고 |
| `decision` | 내용 검토 결과: `approve`, `edit`, `reject` |
| `feedback` | 내용 수정에 반영할 의견 |
| `publication_decision` | 게시 승인 결과: `approve`, `reject` |
| `approved_post` | 두 검토를 모두 승인한 확정 공고 |

`job_input`의 `requirements`에 `job_requirements`를 넣고 나머지는 빈 문자열로 시작하세요. 두 결정은 별도 필드에 저장해, 내용 승인과 게시 승인을 구분합니다.

In [ ]:
# (1) JobState와 초기 입력 job_input을 직접 작성하세요.

### 2. 노드를 하나씩 만듭니다

#### 1번 write_post: 채용 요건으로 공고를 작성합니다

제공한 `post_chain`에 `requirements`를 전달하고 응답의 `.text`를 `post`로 반환하세요.

In [ ]:
# 제공: 채용 요건에 없는 조건은 만들지 않도록 작성 기준 설정
post_prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 요건으로 짧은 한국어 채용공고를 작성하세요. "
     "업무, 필수 역량, 근무 방식, 지원 방법을 구분하세요. "
     "급여와 마감일이 미정이면 미정이라고 밝히고 임의의 금액이나 날짜를 넣지 마세요."),
    ("human", "채용 요건:\n{requirements}"),
])
post_chain = post_prompt | llm

In [ ]:
# (1) 요건을 읽어 공고를 작성하는 노드 생성
def write_post(state: JobState):
    """확정된 채용 요건으로 공고 초안을 만듭니다."""
    response = ...
    return ...

#### 2번 review_post: 공고 내용을 검토합니다

첫 번째 HITL입니다. 아래 요청을 `interrupt()`에 전달하세요.

| 요청 키 | 값 |
|---|---|
| `stage` | `"content_review"` |
| `post` | State의 `post` |
| `actions` | `["approve", "edit", "reject"]` |

재개 응답의 `action`을 `decision`에, `feedback`을 같은 이름의 필드에 저장합니다. 의견이 없으면 빈 문자열을 사용하세요. `stage`는 요청에 붙이는 단계 이름이며 State 필드로 추가하지 않습니다.

In [ ]:
# (2) 내용 검토 요청과 응답을 연결하는 노드 생성
def review_post(state: JobState):
    """공고 내용에 대한 담당자의 선택과 의견을 저장합니다."""
    decision = ...
    return ...

#### 3번 revise_post: 검토 의견으로 공고를 수정합니다

`post_revision_chain`에 `requirements`, `post`, `feedback`을 전달하세요. 수정한 글을 `post`에 넣고 `decision`을 비워 다시 내용 검토를 받습니다.

In [ ]:
# 제공: 확정 요건을 유지하며 사람의 의견을 반영하는 수정 기준
post_revision_prompt = ChatPromptTemplate.from_messages([
    ("system", "채용 요건은 유지하면서 검토 의견에 맞게 공고를 수정하세요. "
     "급여나 마감일이 미정이면 그대로 명시하고, 확정되지 않은 혜택을 추가하지 마세요. "
     "수정한 공고만 출력하세요."),
    ("human", "채용 요건:\n{requirements}\n\n공고:\n{post}\n\n검토 의견:\n{feedback}"),
])
post_revision_chain = post_revision_prompt | llm

In [ ]:
# (3) 내용 검토 의견을 반영하는 노드 생성
def revise_post(state: JobState):
    """확정 요건을 유지하면서 담당자의 의견대로 공고를 수정합니다."""
    response = ...
    return ...

#### 4번 approve_post: 게시 여부를 결정합니다

두 번째 HITL입니다. `interrupt()`에 `stage="publication_approval"`, State의 `post`, `actions=["approve", "reject"]`를 담아 전달하세요. 응답의 `action`을 `publication_decision`으로 반환합니다.

첫 검토의 `decision`은 유지합니다. 게시 반려 시에는 확정하지 않고 종료합니다. 실제 서비스의 담당자 권한 확인은 별도로 구현하며 `stage`가 권한을 검사하지는 않습니다.

In [ ]:
# (4) 내용 승인이 끝난 공고의 게시 여부를 묻는 노드 생성
def approve_post(state: JobState):
    """게시 책임자의 승인 또는 반려를 별도로 저장합니다."""
    decision = ...
    return ...

#### 5번 finalize_post: 두 승인이 끝난 공고를 확정합니다

`finalize_post` 함수를 직접 작성하세요. `post` 값을 `approved_post`로 반환합니다. 두 검토를 모두 승인했을 때만 이 노드에 도달하도록 다음 단계에서 엣지를 연결합니다.

In [ ]:
# (5) finalize_post 노드를 직접 작성하세요.

### 3. 두 검토의 경로를 정합니다

- `route_post`: `decision`이 `approve`이면 `approve`, `edit`이면 `edit`, 그 밖에는 `reject`를 반환합니다.
- `route_publication`: `publication_decision`이 `approve`이면 `approve`, 그 밖에는 `reject`를 반환합니다.

두 함수를 직접 작성하세요. 첫 함수는 내용 검토 결과를, 둘째 함수는 게시 승인 결과를 읽습니다.

In [ ]:
# (1) route_post와 route_publication을 직접 작성하세요.

### 4. 엣지를 연결하고 컴파일합니다

1. `JobState`로 `job_builder`를 만들고 다섯 노드를 등록하세요.
2. `START -> write_post -> review_post`를 연결하세요.
3. `review_post` 뒤에 `route_post`를 연결하세요. `approve`는 **`approve_post`**, `edit`은 `revise_post`, `reject`는 `END`입니다.
4. `approve_post` 뒤에 `route_publication`을 연결하세요. `approve`는 `finalize_post`, `reject`는 `END`입니다.
5. `revise_post -> review_post`, `finalize_post -> END`를 연결하세요.
6. `InMemorySaver()`를 연결해 `job_graph`로 컴파일하세요.

그래프에서 **내용 승인 후 두 번째 검토로 이동하는지** 확인하세요.

In [ ]:
# (1) 상태 스키마로 구성 객체 생성
job_builder = ...

# (2) 다섯 노드와 일반 엣지를 등록하세요.
...

# (3) 두 조건부 엣지와 수정 사이클을 연결하세요.
...

# (4) 대기 상태를 저장할 체크포인터를 연결해 컴파일
job_graph = ...
display(Image(job_graph.get_graph().draw_mermaid_png()))

### 5. 내용을 검토하고 수정 과정을 확인합니다

`job_config`에 `uuid4()`로 만든 새 `thread_id`를 넣고 `job_input`으로 실행하세요. 이 ID는 두 검토가 끝날 때까지 유지합니다. 제공 코드는 첫 중단을 `job_review`에 담아 ID와 `stage`, 공고를 보여 줍니다.

**확인 기준:** `stage`는 `content_review`이고, `approved_post`는 빈 문자열입니다. 이 실습은 순차 실행이므로 한 번에 하나의 중단만 대기합니다.

In [ ]:
# (1) 새 공고 작업을 시작하고 첫 번째 검토에서 대기
job_config = ...
job_paused = ...
job_review = job_graph.get_state(job_config).interrupts[0]
print("중단 ID:", job_review.id)
print("검토 단계:", job_review.value["stage"])
display(Markdown(job_review.value["post"]))
print("확정 공고:", repr(job_paused["approved_post"]))

현재 내용 검토의 ID에 수정 의견을 연결해 `job_stream`을 만드세요.

| `job_graph.stream`에 전달할 항목 | 값 |
|---|---|
| 입력 | `Command(resume={job_review.id: 수정_응답})` |
| 수정 응답 | `action`은 `edit`, `feedback`은 “지원자가 준비할 제출물을 마지막에 두 항목으로 정리해 주세요.” |
| 실행 설정 | `job_config` |
| `stream_mode` | `["messages", "updates"]` |
| `version` | `"v2"` |

초안 작성과 수정에 GPT를 총 2회 호출합니다. 두 승인 단계는 추가 모델 호출이 없습니다.

In [ ]:
# (2) 현재 내용 검토의 ID에 수정 의견을 연결해 스트림 생성
job_review = job_graph.get_state(job_config).interrupts[0]
job_stream = ...

In [ ]:
# 제공: 스트림을 읽으면 수정 노드를 거쳐 내용 검토에서 다시 대기
job_progress = display(Markdown("**진행:** 공고 수정 중"), display_id=True)
job_display = display(Markdown(""), display_id=True)
job_parts = []
for part in job_stream:
    if part["type"] == "messages":
        message, metadata = part["data"]
        if metadata.get("langgraph_node") == "revise_post" and message.text:
            job_parts.append(message.text)
            job_display.update(Markdown("".join(job_parts)))
    elif part["type"] == "updates" and "__interrupt__" in part["data"]:
        job_progress.update(Markdown("**진행:** 수정 완료, 내용 검토 대기"))

# 이전 ID 대신 수정 후 새로 대기한 요청의 ID를 읽음
job_snapshot = job_graph.get_state(job_config)
job_review = job_snapshot.interrupts[0]
print("중단 ID:", job_review.id)
print("검토 단계:", job_review.value["stage"])
job_display.update(Markdown(job_review.value["post"]))
print("확정 공고:", repr(job_snapshot.values["approved_post"]))

### 6. 내용을 승인하고 두 번째 검토를 확인합니다

수정본에 요건에 없는 급여·마감일이 추가되지 않았는지, 수정 의견이 반영됐는지 읽으세요. 확인 후 현재 `job_review.id`에 `{"action": "approve"}`를 연결해 재개하고 결과를 `job_content_result`에 담으세요.

**확인 기준:** 첫 검토의 `decision`은 `approve`, 새 요청의 `stage`는 `publication_approval`입니다. `publication_decision`과 `approved_post`는 아직 빈 문자열입니다. 내용 승인이 끝났지만 게시 승인은 기다리는 상태입니다.

In [ ]:
# (1) 현재 내용 검토에 승인 응답을 전달하세요.
job_review = job_graph.get_state(job_config).interrupts[0]
job_content_result = ...

# 제공: 두 번째 검토 요청을 읽어 단계와 미확정 상태 확인
publication_request = job_graph.get_state(job_config).interrupts[0]
print("내용 검토 결과:", job_content_result["decision"])
print("중단 ID:", publication_request.id)
print("검토 단계:", publication_request.value["stage"])
display(Markdown(publication_request.value["post"]))
print("게시 승인 결과:", repr(job_content_result["publication_decision"]))
print("확정 공고:", repr(job_content_result["approved_post"]))

### 7. 게시 여부를 결정합니다

공고를 확인하고 `publication_action`을 `"approve"` 또는 `"reject"`로 정하세요. 현재 `publication_request.id`에 `{"action": publication_action}`을 연결해 같은 `job_config`로 재개하고 결과를 `job_result`에 담습니다.

| 게시 결정 | 최종 결과 |
|---|---|
| `approve` | 두 결정이 모두 `approve`이며 `approved_post`에 공고가 남음 |
| `reject` | 내용 승인은 유지되지만 게시 결과는 `reject`, `approved_post`는 빈 문자열 |

두 경우 모두 대기 요청과 다음 노드는 비어야 합니다. 다른 결정을 연습하려면 함께 따라하기의 5단계부터 새 작업 ID로 시작하세요. **재개할 때는 항상 현재 대기 목록에서 중단 ID를 읽습니다.**

In [ ]:
# (1) 게시 결정을 정하고 현재 요청에 응답하세요.
publication_action = ...
publication_request = job_graph.get_state(job_config).interrupts[0]
job_result = ...

# 제공: 두 결정과 확정 여부를 함께 확인
print("내용 검토:", job_result["decision"])
print("게시 결정:", job_result["publication_decision"])
if job_result["approved_post"]:
    display(Markdown(job_result["approved_post"]))
else:
    print("게시를 반려하여 확정 공고가 없습니다.")
job_final = job_graph.get_state(job_config)
print("남은 대기:", job_final.interrupts)
print("다음 노드:", job_final.next)

## 학습 내용을 정리합니다

- `interrupt`는 검토값을 전달하고 대기합니다. 같은 `thread_id`의 `Command(resume=...)`으로 사람의 답을 돌려줍니다.
- 재개하면 대기 중이던 노드를 처음부터 실행합니다. 완료한 생성 작업과 외부 처리를 검토 노드에서 분리합니다.
- 조건부 엣지로 승인·수정·반려를 연결하고, 수정 후에는 사람에게 다시 검토받습니다.
- `stage`로 검토 단계를 표시하고 현재 중단 ID에 응답을 연결합니다. 같은 작업의 두 HITL은 같은 `thread_id`로 차례로 재개합니다.
- 내용 검토를 승인해도 게시 승인에서 다시 대기합니다. 두 단계를 모두 승인해야 공고를 확정합니다.
- `stream`은 모드별 실행 정보를, `stream_events(version="v3")`는 한 실행을 읽는 typed projections를 제공합니다.

다음 교안에서는 승인 대기와 대화를 파일에 저장하고 커널 재시작 뒤 복원합니다.